# Quality Control Exploration Using Xarray

In [ ]:
import datetime
import numpy as np
import pandas as pd
from pathlib import Path

import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

In [16]:
# %run HadISD_config.ipynb

# Create an ordered list of all available HadISD station IDs
The  HadISDIndex class contains a method called "get_alL station_ids" that can be used to create a list of all available station IDS. This list can be used to select, for example, the first_ten stations from all stations. It can then be passed used to select data from within the pipeline. It's also possible to pass a single station ID as a string to retrieve data for a single station.

TODO:
- It would be good if a range of stations can be provided, or a boundary with stations inside

In [ ]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids()
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")
first_ten_stations = all_stations_ordered[:10]
print(first_ten_stations)

In [ ]:
# Hadisd accepts "all" as an argument to get all stations.
# A single station can also be specified, e.g. "010010-99999".
# Or, a list of station IDs can be passed.

data_prep_pipe = petpipe.Pipeline(
    petdata.archive.hadisd(station = first_ten_stations), #, variables = ["total_cloud_cover", "temperatures", "flagged_obs"]), # Commented out variable selection whilst looking into bug in BaseTransforms
    SqueezeStationCoordinates(), # Squeeze coordinates to 1D if they are 2D: Remove redundant dimensions
    # petdata.transforms.variables.Drop("flagged_obs"), # Drop the flagged_obs variable as it is not needed for ML an might even introduce bias
    # FeatureTargetSplit(target_vars="quality_control_flags"), # Split dataset into features and target variable
)
data_prep_pipe

In [ ]:
y = data_prep_pipe["1969-01-01T07"]
y

In [ ]:
x

In [ ]:
qc = y["quality_control_flags"].values


In [ ]:
# # Suppose you want test 0:
# qc_test_0 = qc[:, :, 0]  # shape: (station, time)
# qc_df = pd.DataFrame(qc_test_0, index=y["quality_control_flags"].coords["station"].values, columns=x.time.values)
# qc_df.head()

In [ ]:
# # Convert to pandas DataFrame for easier manipulation
# qc_df = pd.DataFrame(qc, columns=y["quality_control_flags"].dims[1:], index=x.time.values)
# # Display the first few rows of the DataFrame
# qc_df.head()

In [ ]:
print("Unique QC flag values:", np.unique(qc[~np.isnan(qc)]))
print("QC flag shape:", qc.shape)

In [ ]:
for i in range(qc.shape[2]):  # for each test
    unique_vals = np.unique(qc[:, :, i][~np.isnan(qc[:, :, i])])
    print(f"Test {i}: unique QC flag values: {unique_vals}")

In [ ]:
# Sum of flags per test (across all stations and times)
qc_sum_per_test = np.nansum(qc, axis=(0, 1))
print("QC fails per test:", qc_sum_per_test)

# Sum per station
qc_sum_per_station = np.nansum(qc, axis=(1, 2))
print("QC fails per station:", qc_sum_per_station)

In [ ]:
max_fails = 0
max_test = None

for i in range(0, 71):
    failed = np.where(qc[:, :, i] == 1)
    num_fails = len(failed[0])
    print(f"Stations with fails for test {i}:", failed[0])
    print(f"Times with fails for test {i}:", failed[1])
    print(f"Number of fails for test {i}: {num_fails}")
    if num_fails > max_fails:
        max_fails = num_fails
        max_test = i

print(f"Test with most fails: {max_test} ({max_fails} fails)")


In [ ]:
# Stations with fails for test 12: [0]
# Times with fails for test 12: [826]

# for qc, test 12, time 826, station 0, print the value of the test
print("QC value for test 12, time 826, station 0:", qc[0, 826, 12])